# Logistic Regression - Script Ordered Storyboard

This notebook mirrors the `script.md` narration for Chapter 1 (The Sigmoid).

Each scene exports either a PNG or GIF into `renders/` so shots can be sequenced directly in voice-over order.

- We intentionally skip the truck/car recap visuals.
- The required students `(3h study, 2h exam, passed)` and `(2h study, 3h exam, failed)` are included.
- Parallel lines at `ST - EL = 1, 2, 3` (and negatives) tie to the narrated `60%`, `80%`, `100%` pass proportions.


In [15]:
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D
from PIL import Image, ImageDraw

np.random.seed(7)

OUTPUT_DIR = Path("renders")
OUTPUT_DIR.mkdir(exist_ok=True)

# Typography / padding (single source of truth for exports)
FONT_SIZE = 11
AXIS_LABEL_SIZE = 12
LEGEND_SIZE = 10
TITLE_SIZE = 12
NOTE_SIZE = 10
ANNOT_SIZE = 9
SAVE_PAD_INCHES = 0.12

plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelpad": 8.0,
        "axes.titlepad": 10.0,
        "legend.fontsize": LEGEND_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.96,
        "legend.borderaxespad": 0.55,
        "legend.labelspacing": 0.35,
        "legend.handlelength": 1.35,
        "legend.handletextpad": 0.65,
        "savefig.pad_inches": SAVE_PAD_INCHES,
    }
)


PASS_COLOR = "#2ca02c"
FAIL_COLOR = "#d62728"
NEUTRAL_COLOR = "#4f4f4f"

CHECK_ICON_PATH = OUTPUT_DIR / "check.png"
CROSS_ICON_PATH = OUTPUT_DIR / "cross.png"


def _ensure_outcome_icons(size=96, line_width=12):
    if not CHECK_ICON_PATH.exists():
        img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
        draw = ImageDraw.Draw(img)
        draw.line([(20, 55), (42, 76), (78, 24)], fill=(44, 160, 44, 255), width=line_width)
        img.save(CHECK_ICON_PATH)

    if not CROSS_ICON_PATH.exists():
        img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
        draw = ImageDraw.Draw(img)
        draw.line([(24, 24), (72, 72)], fill=(214, 39, 40, 255), width=line_width)
        draw.line([(72, 24), (24, 72)], fill=(214, 39, 40, 255), width=line_width)
        img.save(CROSS_ICON_PATH)


_ensure_outcome_icons()
CHECK_ICON = np.asarray(Image.open(CHECK_ICON_PATH).convert("RGBA"))
CROSS_ICON = np.asarray(Image.open(CROSS_ICON_PATH).convert("RGBA"))


def add_outcome_icon(ax, x, y_value, passed, zoom=0.16, alpha=1.0):
    image_array = CHECK_ICON if passed else CROSS_ICON
    icon = OffsetImage(image_array, zoom=zoom)
    icon.set_alpha(alpha)
    ab = AnnotationBbox(icon, (x, y_value), frameon=False)
    ax.add_artist(ab)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def fig_to_image(fig, dpi=140):
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def save_gif(images, filename, duration=40):
    if not images:
        raise ValueError("images list is empty")
    images[0].save(
        OUTPUT_DIR / filename,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
    )


In [ ]:
# Two-stage dataset design:
# 1) Before "that's not realistic": linearly separable points.
# 2) After that line: add symmetric noisy students to break separability.

separable_points = [
    # Fail class (left of boundary, z=study-exam < 0)
    (2, 3, 0),  # required fail example
    (4, 5, 0), (5, 6, 0),
    (1, 3, 0), (2, 4, 0), (4, 6, 0),
    (1, 4, 0), (3, 6, 0), (1, 6, 0),


    # Pass class (right of boundary, z > 0)
    (3, 2, 1),  # required pass example
    (5, 4, 1), (6, 5, 1),                    # z=+1 -> 3 pass
    (4, 2, 1), (6, 4, 1), (3, 1, 1),  # z=+2 -> 4 pass
    (4, 1, 1), (5, 2, 1), (6, 3, 1),  # z=+3 -> 4 pass
    (6, 1, 1),
]

# Added noisy students (symmetric around z=0, on parallel lines)
# to break strict linear separability after "but that's not realistic".
noisy_symmetric_points = [
    # symmetric pairs on +/-1
    (2, 1, 0),  # z=+1 but fails
    (1, 2, 1),  # z=-1 but passes
    (3, 4, 1),  # z=-2 but fails
    (4, 3, 0),  # z=+2 but passes

    # symmetric pair on +/-2
    (3, 5, 1),  # z=+2 but passes
    (5, 3, 0),  # z=-2 but fails
]

realistic_points = separable_points + noisy_symmetric_points


def unpack_points(point_list):
    arr = np.array(point_list, dtype=float)
    s = arr[:, 0]
    e = arr[:, 1]
    labels = arr[:, 2].astype(int)
    diff = s - e
    return s, e, labels, diff


study_sep, exam_sep, y_sep, z_sep = unpack_points(separable_points)
study_real, exam_real, y_real, z_real = unpack_points(realistic_points)

xlim = (0, 7)
ylim = (0, 7)

midpoint_shift = (z_sep[y_sep == 0].max() + z_sep[y_sep == 1].min()) / 2.0

print(f"Separable students: {len(separable_points)}")
print(f"Realistic students: {len(realistic_points)}")
print("Required pass example present:", np.any((study_sep == 3) & (exam_sep == 2) & (y_sep == 1)))
print("Required fail example present:", np.any((study_sep == 2) & (exam_sep == 3) & (y_sep == 0)))
print("Threshold midpoint between classes (separable stage):", midpoint_shift)


Separable students: 19
Realistic students: 25
Required pass example present: True
Required fail example present: True
Threshold midpoint between classes (separable stage): 0.0


In [17]:
def draw_dataset(ax, study_vals, exam_vals, labels, mask=None, alpha=0.95, title=None):
    if mask is None:
        mask = np.ones(len(study_vals), dtype=bool)

    for i in np.where(mask)[0]:
        add_outcome_icon(ax, study_vals[i], exam_vals[i], passed=bool(labels[i]), alpha=alpha)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def add_combined_legend(ax, loc="upper left"):
    line_handles, line_labels = ax.get_legend_handles_labels()
    merged_handles = []
    merged_labels = []
    seen = set()

    for h, lbl in zip(line_handles, line_labels):
        if lbl and lbl not in seen:
            merged_handles.append(h)
            merged_labels.append(lbl)
            seen.add(lbl)

    if merged_handles:
        ax.legend(
            handles=merged_handles,
            labels=merged_labels,
            loc=loc,
            prop={"size": LEGEND_SIZE},
        )


def add_threshold_line(ax, shift=0.0, label=None, style="--", color=NEUTRAL_COLOR, linewidth=1.0):
    x = np.linspace(*xlim, 200)
    y_line = x - shift
    ax.plot(x, y_line, style, color=color, linewidth=linewidth, label=label)



def draw_axes_only(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.grid(alpha=0.2)
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)


def shade_pass_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    top = np.minimum(y_line, yb)
    bot = np.full_like(xs, ya)
    ax.fill_between(xs, bot, top, where=(top > bot), alpha=alpha, color=PASS_COLOR, linewidth=0, zorder=0)


def shade_fail_half(ax, shift=0.0, alpha=0.22):
    xa, xb = xlim
    ya, yb = ylim
    xs = np.linspace(xa, xb, 400)
    y_line = xs - shift
    bot2 = np.maximum(y_line, ya)
    ax.fill_between(xs, bot2, yb, where=(yb > bot2), alpha=alpha, color=FAIL_COLOR, linewidth=0, zorder=0)


def draw_z_axis_panel(ax, z_value, label):
    ax.axhline(0, color="black", linewidth=1)
    ax.scatter([z_value], [0], color="#ff7f0e", s=120, zorder=5)
    ax.text(z_value + 0.05, 0.08, label, fontsize=NOTE_SIZE)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.set_xlabel("ST - EL", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.tick_params(axis="x", which="major", labelsize=FONT_SIZE)
    ax.grid(alpha=0.15)


def save_fig(fig, filename):
    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        pad_inches=SAVE_PAD_INCHES,
    )
    plt.close(fig)


## Scene 1 - Introduce the dataset

Script alignment: survey students, collect study time, exam length, and pass/fail outcome.

The **next code cell** exports `00`–`14` (axes-only, staged collection order, boundary crossing, then threshold visuals). The two cells after that export the full overview plots (`15`–`16`).


In [18]:
# Intro montage (exports 00-14): axes-only, staged build, crossing story, threshold pedagogy.
rng_build = np.random.default_rng(7)
n_sep = len(study_sep)

# ---- 00: axes only (no data) ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_axes_only(ax)
save_fig(fig, "00_axes_only.png")

# ---- 01: dataset build — (3,2,1) then (2,3,0) then random order ----
idx_pass_req = int(np.where((study_sep == 3) & (exam_sep == 2) & (y_sep == 1))[0][0])
idx_fail_req = int(np.where((study_sep == 2) & (exam_sep == 3) & (y_sep == 0))[0][0])
rest = [i for i in range(n_sep) if i not in (idx_pass_req, idx_fail_req)]
rng_build.shuffle(rest)
reveal_order = [idx_pass_req, idx_fail_req] + rest

frames = []
# Hold empty axes, then one new point per step; first two points linger much longer.
DATASET_EMPTY_HOLD = 24
DATASET_FIRST_TWO_HOLD = 48
DATASET_REST_HOLD = 2


def _append_dataset_build_frames(mask, n_repeat):
    for _ in range(n_repeat):
        fig, ax = plt.subplots(figsize=(8, 6))
        draw_dataset(ax, study_sep, exam_sep, y_sep, mask=mask)
        add_combined_legend(ax, loc="upper left")
        frames.append(fig_to_image(fig))


mask = np.zeros(n_sep, dtype=bool)
_append_dataset_build_frames(mask, DATASET_EMPTY_HOLD)

for k in range(1, n_sep + 1):
    mask = np.zeros(n_sep, dtype=bool)
    mask[reveal_order[:k]] = True
    hold = DATASET_FIRST_TWO_HOLD if k <= 2 else DATASET_REST_HOLD
    _append_dataset_build_frames(mask, hold)

save_gif(frames, "01_dataset_build.gif", duration=35)

# ---- 02: failed point pushed right (more study); threshold/boundary not drawn ----
EXAM_ANIM = 3.0
ST_START, ST_END = 2.0, 3.25
N_ANIM = 55
PUSH_HOLD = 5
st_values = np.concatenate([np.linspace(ST_START, ST_END, N_ANIM), np.full(PUSH_HOLD, ST_END)])
static_mask = np.ones(n_sep, dtype=bool)
static_mask[idx_fail_req] = False
frames = []
for st in st_values:
    passed = (st - EXAM_ANIM) > 1e-6
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep, mask=static_mask, alpha=0.28)
    add_outcome_icon(ax, st, EXAM_ANIM, passed=passed, zoom=0.24, alpha=1.0)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "02_fail_point_crosses_threshold.gif", duration=42)

# ---- 03: threshold, unlabeled ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "03_threshold_unlabeled.png")

# ---- 04: unlabeled threshold + pass half green ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "04_threshold_green_right.png")

# ---- 05: unlabeled threshold + fail half red ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "05_threshold_red_left.png")

# ---- 06: threshold drifts slightly (still unlabeled) ----
drift = np.concatenate([
    np.linspace(0, 0.12, 18),
    np.linspace(0.12, -0.12, 22),
    np.linspace(-0.12, 0, 18),
])
frames = []
for sh in drift:
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep)
    add_threshold_line(ax, shift=midpoint_shift + sh, label=None, color="black", linewidth=1)
    add_combined_legend(ax, loc="upper left")
    frames.append(fig_to_image(fig))
save_gif(frames, "06_threshold_micro_drift.gif", duration=40)

# ---- 07: add (3,3) pass; threshold shifted so classes still separated ----
SHIFT_FOR_33 = -0.28
st_33 = np.append(study_sep, 3.0)
ex_33 = np.append(exam_sep, 3.0)
y_33 = np.append(y_sep, 1)
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, st_33, ex_33, y_33)
add_threshold_line(ax, shift=midpoint_shift + SHIFT_FOR_33, label=None, color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "07_dataset_extra_pass_shifted_threshold.png")

# ---- 08: (3,3) pass + (4,3) fail, no threshold ----
st_3343 = np.append(st_33, 4.0)
ex_3343 = np.append(ex_33, 3.0)
y_3343 = np.append(y_33, 0)
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, st_3343, ex_3343, y_3343)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "08_dataset_two_extra_no_threshold.png")

# ---- 09: threshold labeled ST = EL ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "09_threshold_labeled_st_eq_el.png")

# ---- 10: ST = EL + black circle at (3,3) ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.plot([3], [3], marker="o", markersize=11, markerfacecolor="black", markeredgecolor="black", linestyle="none", zorder=6)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "10_threshold_st_eq_el_dot_33.png")

# ---- 11: labeled threshold + green pass half + ST > EL ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_pass_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(5.15, 0.95, "ST > EL", fontsize=AXIS_LABEL_SIZE, color="#145214", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "11_threshold_green_labeled_st_gt_el.png")

# ---- 12: labeled threshold + red fail half + ST < EL ----
fig, ax = plt.subplots(figsize=(8, 6))
shade_fail_half(ax, shift=midpoint_shift)
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST = EL", color="black", linewidth=1)
ax.text(0.55, 5.35, "ST < EL", fontsize=AXIS_LABEL_SIZE, color="#7a1212", fontweight="bold")
add_combined_legend(ax, loc="upper left")
save_fig(fig, "12_threshold_red_labeled_st_lt_el.png")

# ---- 13: line labeled ST - EL = EL - EL ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = EL - EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "13_threshold_label_st_minus_el_eq_el_minus_el.png")

# ---- 14: line labeled ST - EL = 0 ----
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label="ST - EL = 0", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "14_threshold_label_st_minus_el_eq_0.png")


In [19]:
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"ST = EL", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "15_dataset_overview.png")


In [20]:
fig, ax = plt.subplots(figsize=(8, 6))
draw_dataset(ax, study_sep, exam_sep, y_sep)
add_threshold_line(ax, shift=midpoint_shift, label=f"threshold: ST - EL = {midpoint_shift:.1f}", color="black", linewidth=1)
add_combined_legend(ax, loc="upper left")
save_fig(fig, "16_dataset_overview_labeled.png")


In [21]:
# Superseded: dataset build order is now `01_dataset_build.gif` in the intro cell above.


## Scene 2 - Required concrete students

Script alignment: the separable dataset still includes the **3h / 2h exam (pass)** and **2h / 3h exam (fail)** examples in the main plots.

Dedicated zoom exports for those two students were removed; use intro `01_dataset_build.gif` and overview PNGs if you need them on camera.


In [22]:
# Removed exports 17–20 (per-student plane + z-line focus plots).


## Scene 3 - Midpoint threshold (narration gap)

The animated threshold search and static midpoint line exports were removed (`21` / `22`).

The story continues at **parallel difference lines** (`17` / `18`) with `z = ST - EL` still centered at **0** for the separable stage.


In [23]:
# Removed: `21_threshold_guesses.gif` and `22_threshold_line.png` (threshold search animation + static line).


## Scene 4 - Parallel lines (separate additive plots)

Script alignment: each line for fixed `ST - EL` appears one-by-one in additive order.


In [24]:
line_specs = [
    (0, "black", "ST - EL = 0 (threshold)"),
    (1, "#1f77b4", "ST - EL = 1"),
    (-1, "#9467bd", "ST - EL = -1"),
    (2, "#17becf", "ST - EL = 2"),
    (3, "#2ca02c", "ST - EL = 3"),
]

PARALLEL_HOLD = 4

frames = []
for step in range(1, len(line_specs) + 1):
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_dataset(ax, study_sep, exam_sep, y_sep)

    for shift, color, label in line_specs[:step]:
        add_threshold_line(ax, shift=shift, label=label, color=color, linewidth=1)

    add_combined_legend(ax, loc="upper left")
    file_name = f"17_parallel_lines_step_{step:02d}.png"
    save_fig(fig, file_name)

    frame = Image.open(OUTPUT_DIR / file_name).convert("RGB")
    for _ in range(PARALLEL_HOLD):
        frames.append(frame)

save_gif(frames, "18_parallel_lines_additive.gif", duration=40)


## Scene 5 - "But that's not realistic" (add symmetric noisy students)

Script alignment:
- realistic students break perfect separability while keeping the same midpoint threshold
- `19_not_realistic_transition.gif`: single plane, **no labels**, threshold fixed, points appear one at a time
- `20_proportions_60_80_100.gif`: parallel lines **one at a time**; first line counts with a grow/shrink emphasis; each line shows **positives/total = pass rate** next to it
- narrated proportions for `ST - EL = 1, 2, 3` remain `60%`, `80%`, `100%`


In [25]:
# Scene 5 — "not realistic": single-plane GIF, threshold fixed, points appear one-by-one (no axis labels / no legend).
SC5_MS = 40
HOLD_EMPTY = 18
HOLD_PER_POINT = 2
n_r = len(study_real)


def _strip_xy(ax):
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelbottom=False, labelleft=False)


frames = []
for _ in range(HOLD_EMPTY):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
    _strip_xy(ax)
    frames.append(fig_to_image(fig))

reveal_idx = np.lexsort((exam_real, study_real))
for k in range(1, n_r + 1):
    mask = np.zeros(n_r, dtype=bool)
    mask[reveal_idx[:k]] = True
    for _ in range(HOLD_PER_POINT):
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.grid(alpha=0.12)
        add_threshold_line(ax, shift=midpoint_shift, label=None, color="black", linewidth=1)
        draw_dataset(ax, study_real, exam_real, y_real, mask=mask, alpha=0.95)
        _strip_xy(ax)
        frames.append(fig_to_image(fig))

save_gif(frames, "19_not_realistic_transition.gif", duration=SC5_MS)

# --- Line-level proportions (realistic): one GIF, lines appear in order; first line counts with grow/shrink. ---
targets = [1, 2, 3]
line_colors = ["#1f77b4", "#17becf", "#2ca02c"]
prop = []
counts = []
for value in targets:
    m = z_real == value
    prop.append(float(y_real[m].mean()) if m.any() else 0.0)
    counts.append(int(m.sum()))

frames = []


def _prob_xy_for_line(shift, x_hint=5.35):
    y_line = x_hint - shift
    return x_hint + 0.12, y_line + 0.28


def _emit_hold(img, n):
    for _ in range(n):
        frames.append(img.copy())


lines_pairs = []

for _ in range(10):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.12)
    _strip_xy(ax)
    frames.append(fig_to_image(fig))

for ti, (value, lcol) in enumerate(zip(targets, line_colors)):
    lines_pairs.append((value, lcol))

    for _ in range(5):
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.grid(alpha=0.12)
        for sh, col in lines_pairs:
            add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
        mask_prev = np.zeros(n_r, dtype=bool)
        for sh, _ in lines_pairs[:-1]:
            mask_prev |= z_real == sh
        draw_dataset(ax, study_real, exam_real, y_real, mask=mask_prev, alpha=0.95)
        _strip_xy(ax)
        frames.append(fig_to_image(fig))

    if value == 1:
        idx1 = np.where(z_real == 1)[0]
        idx1 = idx1[np.argsort(study_real[idx1], kind="mergesort")]
        n1 = len(idx1)
        for j in range(0, n1 + 1):
            grow_states = (True, False) if j > 0 else (True,)
            for grow in grow_states:
                fig, ax = plt.subplots(figsize=(8, 6))
                ax.set_xlim(*xlim)
                ax.set_ylim(*ylim)
                ax.grid(alpha=0.12)
                for sh, col in lines_pairs:
                    add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
                for k, ii in enumerate(idx1):
                    if k >= j:
                        break
                    last = k == j - 1
                    zm = 0.26 if last and grow else 0.16
                    add_outcome_icon(ax, study_real[ii], exam_real[ii], passed=bool(y_real[ii]), zoom=zm, alpha=0.95)
                if j > 0:
                    pos = int(y_real[idx1[:j]].sum())
                    tx, ty = _prob_xy_for_line(1)
                    ax.text(tx, ty, f"{pos}/{j}", fontsize=NOTE_SIZE, color="black")
                _strip_xy(ax)
                im = fig_to_image(fig)
                frames.append(im)
                _emit_hold(im, 1 if grow else 0)
        p1, c1 = prop[ti], counts[ti]
        pos1 = int(round(p1 * c1))
        for _ in range(10):
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.grid(alpha=0.12)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            for ii in idx1:
                add_outcome_icon(ax, study_real[ii], exam_real[ii], passed=bool(y_real[ii]), zoom=0.16, alpha=0.95)
            tx, ty = _prob_xy_for_line(1)
            ax.text(tx, ty, f"{pos1}/{c1} = {100 * p1:.0f}%", fontsize=NOTE_SIZE, color="black")
            _strip_xy(ax)
            frames.append(fig_to_image(fig))
    else:
        idxv = np.where(z_real == value)[0]
        idxv = idxv[np.argsort(study_real[idxv], kind="mergesort")]
        for _ in range(8):
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.grid(alpha=0.12)
            for sh, col in lines_pairs:
                add_threshold_line(ax, shift=sh, label=None, color=col, linewidth=1)
            mask_prev = np.zeros(n_r, dtype=bool)
            for sh, _ in lines_pairs[:-1]:
                mask_prev |= z_real == sh
            draw_dataset(ax, study_real, exam_real, y_real, mask=mask_prev, alpha=0.95)
            for ii in idxv:
                add_outcome_icon(ax, study_real[ii], exam_real[ii], passed=bool(y_real[ii]), zoom=0.16, alpha=0.95)
            pv, cv = prop[ti], counts[ti]
            posv = int(round(pv * cv))
            tx, ty = _prob_xy_for_line(value)
            ax.text(tx, ty, f"{posv}/{cv} = {100 * pv:.0f}%", fontsize=NOTE_SIZE, color="black")
            _strip_xy(ax)
            frames.append(fig_to_image(fig))

save_gif(frames, "20_proportions_60_80_100.gif", duration=SC5_MS)

print("ST-EL=+1 pass rate:", prop[0])
print("ST-EL=+2 pass rate:", prop[1])
print("ST-EL=+3 pass rate:", prop[2])


ST-EL=+1 pass rate: 0.8
ST-EL=+2 pass rate: 0.75
ST-EL=+3 pass rate: 1.0


## Scene 6 - Student between 60% and 80%

Script alignment: a student with `ST - EL` between **+1** and **+2** should map to a probability between **0.6** and **0.8**.

`21_between_60_and_80.png` is now the **study vs exam** plane (reference lines at `ST - EL = 0, 1, 2`) with a **black marker** at **(3.5, 2)** (`z = 1.5`).


In [26]:
# Scene 6 — "between" student: ST–EL plane with z = +1 / +2 guides + black marker at (3.5, 2); no labels.
z_anchor = np.array([1.0, 2.0, 3.0])
p_anchor = np.array([0.6, 0.8, 1.0])

query_z = 1.5
query_p = np.interp(query_z, z_anchor, p_anchor)

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.grid(alpha=0.12)
add_threshold_line(ax, shift=0, label=None, color="black", linewidth=1)
add_threshold_line(ax, shift=1, label=None, color="#bbbbbb", linewidth=1)
add_threshold_line(ax, shift=2, label=None, color="#bbbbbb", linewidth=1)
draw_dataset(ax, study_real, exam_real, y_real, alpha=0.32)
ax.plot(
    [3.5],
    [2.0],
    marker="o",
    markersize=11,
    markerfacecolor="black",
    markeredgecolor="black",
    linestyle="none",
    zorder=15,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(labelbottom=False, labelleft=False)
save_fig(fig, "21_between_60_and_80.png")

print(f"Interpolated probability for ST-EL={query_z}: {query_p:.2f}")


Interpolated probability for ST-EL=1.5: 0.70


## Scene 7 - Exponential step-by-step ($2^x$, step size 1 in $x$)

Script alignment (Chapter 1 script):
- moving **one unit left** in $x$ (so $\Delta x=-1$) must shrink the positive output while leaving room for all negatives
- the consistent rule is **multiply by $\frac{1}{2}$** each step — that is exactly **$2^x$** on integer steps

`22_exponential_mapping.gif` is **only** that local $2^x$ panel: **no** axis tick labels, with on-screen text spelling out **input step size** and the **half-factor** on the output.

`23_exponential_global.png` is a static wide view of the same **$2^z$** map for reference.


In [27]:
# 22 — exponential idea per script: **unit steps in x** (Δx = −1), **output halves each step** → plot **2^x** (not e^x). Single panel, no axis labels/ticks.
step_values = np.arange(0, -7, -1)
EXP_STEP_HOLD = 5
DELTA_X = -1  # script: "step of size 1" to the left

frames = []


def _strip(ax):
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    ax.tick_params(labelbottom=False, labelleft=False)


for step_idx, x_cur in enumerate(step_values):
    x_prev = x_cur - DELTA_X  # previous (to the right) integer when Δx = -1
    y_cur = float(np.power(2.0, x_cur))
    y_prev = float(np.power(2.0, x_prev)) if step_idx > 0 else None

    for _ in range(EXP_STEP_HOLD):
        fig, ax = plt.subplots(figsize=(8, 5.2))
        xs = np.linspace(-6.6, 1.4, 520)
        ax.plot(xs, np.power(2.0, xs), color="#1f77b4", linewidth=2)
        ax.scatter([x_cur], [y_cur], color="#ff7f0e", s=130, zorder=5, clip_on=False)
        ax.set_xlim(-6.6, 1.4)
        ax.set_ylim(0, 1.15)
        ax.grid(alpha=0.2)
        _strip(ax)

        ax.text(0.04, 0.93, r"$f(x)=2^x$", transform=ax.transAxes, fontsize=TITLE_SIZE, va="top")

        if step_idx == 0:
            body = (
                "Input step size in $x$: $1$ (script: one unit per move toward negatives).\n"
                "Output at $x=0$: $2^0=1$.\n"
                r"Next move: still step size $1$ in $x$, so $\Delta x=-1$."
            )
        else:
            body = (
                r"Input: step size $1$ in $x$ $\Rightarrow$ $\Delta x=-1$ (from $"
                + f"{x_prev:g}"
                + r"$ to $"
                + f"{x_cur:g}"
                + r"$)."
                + "\n"
                r"Each such step multiplies $2^x$ by $\frac{1}{2}$."
                + "\n"
                + rf"Check: $2^{{{x_cur:g}}}={y_cur:.4f}=\frac{{1}}{{2}}\cdot 2^{{{x_prev:g}}}=\frac{{1}}{{2}}\cdot {y_prev:.4f}$."
            )
        ax.text(0.04, 0.78, body, transform=ax.transAxes, fontsize=NOTE_SIZE, va="top", linespacing=1.35)

        frames.append(fig_to_image(fig))

save_gif(frames, "22_exponential_mapping.gif", duration=70)

# Static reference: same 2^x map (still labeled for the separate reference slide — optional strip)
z_big = np.linspace(-8, 8, 800)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_big, np.power(2.0, z_big), color="#1f77b4", linewidth=2.3)
ax.set_xlim(-8, 8)
ax.set_ylim(0, float(np.power(2.0, 8)) * 1.02)
ax.set_xlabel("ST - EL")
ax.set_ylabel(r"$2^{(ST-EL)}$")
ax.set_title(r"Map $z \mapsto 2^z$ (reference)")
ax.grid(alpha=0.2)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
save_fig(fig, "23_exponential_global.png")


## Scene 8 - Sigmoid in 3D (pass/fail mirrored surfaces)

Script alignment:
- map any `ST - EL` value to pass probability with sigmoid
- show mirrored fail probability
- present both in 3D style


In [28]:
import matplotlib as mpl
st_axis = np.linspace(xlim[0], xlim[1], 220)
el_axis = np.linspace(ylim[0], ylim[1], 220)
ST, EL = np.meshgrid(st_axis, el_axis)
DIFF = ST - EL
P_PASS = sigmoid(DIFF)
P_FAIL = sigmoid(-DIFF)
cvals  = [0, .5, 1]
colors = ['red', 'white', 'green']
norm=plt.Normalize(min(cvals),max(cvals))
tuples = list(zip(map(norm,cvals), colors))
CMAP = mpl.colors.LinearSegmentedColormap.from_list("", tuples, 100)

# Threshold curve where ST=EL and sigmoid(0)=1/2
diag_line = np.linspace(max(xlim[0], ylim[0]), min(xlim[1], ylim[1]), 220)
threshold_half = np.full_like(diag_line, 0.5)

# Reference cross-section for readability (fixed exam length)
el_ref = np.full_like(st_axis, np.mean(ylim))

all_study = study_real
all_exam = exam_real
all_diff = all_study - all_exam
pass_idx = y_real == 1
fail_idx = y_real == 0

# Match the red/green palette used in neural-networks.ipynb (21.gif section)
SIG_PASS_COLOR = "green"
SIG_FAIL_COLOR = "red"


def style_sigmoid_axes(ax, az):
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.tick_params(axis="x", labelsize=FONT_SIZE)
    ax.tick_params(axis="y", labelsize=FONT_SIZE)
    ax.tick_params(axis="z", labelsize=FONT_SIZE)
    ax.set_xlabel("Study Time (ST)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam Length (EL)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=26, azim=az)


def scatter_whole_data_by_class(ax, prob_values):
    ax.scatter(all_study[pass_idx], all_exam[pass_idx], prob_values[pass_idx], c=SIG_PASS_COLOR, s=30, alpha=0.95, depthshade=False)
    ax.scatter(all_study[fail_idx], all_exam[fail_idx], prob_values[fail_idx], c=SIG_FAIL_COLOR, s=30, alpha=0.95, depthshade=False)


ROTATION_FRAMES = 120
ROTATION_MS = 32
rotation_azimuths = np.linspace(25, 345, ROTATION_FRAMES)


# Animated 3D rotation around pass/fail sigmoid surfaces (correct ST/EL axes)
frames = []
for az in rotation_azimuths:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)

    style_sigmoid_axes(ax, az)
    ax.text2D(0.02, 0.95, r"$\sigma(x)=\frac{1}{1+e^{-x}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
    frames.append(fig_to_image(fig, dpi=150))

save_gif(frames, "24_sigmoid_and_mirror.gif", duration=ROTATION_MS)

# Static 3D snapshot
fig = plt.figure(figsize=(10.5, 5.4))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(ST, EL, P_PASS, alpha=0.30, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
ax.plot_surface(ST, EL, P_FAIL, alpha=0.24, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
style_sigmoid_axes(ax, 58)
ax.text2D(0.02, 0.95, r"$\sigma(x)=\frac{1}{1+e^{-x}}$", transform=ax.transAxes, fontsize=TITLE_SIZE)
save_fig(fig, "25_sigmoid_static.png")

# Additional requested 3D sigmoid plots (no formula text)
from matplotlib.colors import LinearSegmentedColormap

extra_angles = rotation_azimuths

# 1) Green-only pass sigmoid with whole data (class-colored)
frames = []
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_PASS, alpha=0.26, color=SIG_PASS_COLOR, linewidth=0, antialiased=True)
    ax.plot(st_axis, el_ref, sigmoid(st_axis - el_ref), color="#0b6c0b", linewidth=2.4)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "26_sigmoid_pass_green.gif", duration=ROTATION_MS)

# 2) Red-only fail sigmoid with whole data (class-colored)
frames = []
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, P_FAIL, alpha=0.26, color=SIG_FAIL_COLOR, linewidth=0, antialiased=True)
    ax.plot(st_axis, el_ref, sigmoid(-(st_axis - el_ref)), color="#8b1b1b", linewidth=2.4)
    scatter_whole_data_by_class(ax, sigmoid(-all_diff))
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "27_sigmoid_fail_red.gif", duration=ROTATION_MS)


frames = []
for az in extra_angles:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    style_sigmoid_axes(ax, az)
    frames.append(fig_to_image(fig, dpi=150))
save_gif(frames, "28_sigmoid_pass_colormap.gif", duration=ROTATION_MS)

# 4) Tilt up to top-down so it connects to prior 2D dataset views,
# then rotate counterclockwise by 32 degrees at the top.
TOPDOWN_FRAMES = 90
TOPDOWN_TURN_FRAMES = 40
TOPDOWN_END_HOLD_FRAMES = 12  # duplicate final view after rotation
topdown_elevations = np.linspace(26, 89, TOPDOWN_FRAMES)
topdown_azimuth = 180 + 58
counterclockwise_turn = np.linspace(topdown_azimuth, topdown_azimuth + 32, TOPDOWN_TURN_FRAMES)
frames = []

# Phase 1: tilt upward
for elev in topdown_elevations:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Study Time (ST)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam Length (EL)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.zaxis.label.set_visible(elev < topdown_elevations[-1])
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=elev, azim=topdown_azimuth)
    frames.append(fig_to_image(fig, dpi=150))

# Phase 2: at top, rotate counterclockwise 32 degrees
for az in counterclockwise_turn:
    fig = plt.figure(figsize=(10.5, 5.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(ST, EL, sigmoid(ST - EL), alpha=0.32, cmap=CMAP, linewidth=0, antialiased=True, shade=False)
    scatter_whole_data_by_class(ax, sigmoid(all_diff))
    ax.plot(diag_line, diag_line, threshold_half, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Study Time (ST)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam Length (EL)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_zlabel("Sigmoid Probability", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    ax.zaxis.label.set_visible(False)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0, 1)
    ax.view_init(elev=89, azim=az)
    frames.append(fig_to_image(fig, dpi=150))

hold = frames[-1]
for _ in range(TOPDOWN_END_HOLD_FRAMES):
    frames.append(hold.copy())

save_gif(frames, "29_sigmoid_colormap_to_topdown.gif", duration=ROTATION_MS)

# 5) Top-down 2D colormap counterpart
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(ST, EL, sigmoid(ST - EL), levels=120, cmap=CMAP, vmin=0, vmax=1)
ax.plot(diag_line, diag_line, color="black", linestyle="--", linewidth=1)

for s, e, lbl in zip(all_study, all_exam, y_real):
    add_outcome_icon(ax, s, e, passed=bool(lbl), zoom=0.16, alpha=0.95)

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel("Study Time (ST)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.set_ylabel("Exam Length (EL)", fontsize=AXIS_LABEL_SIZE, labelpad=8)
ax.grid(alpha=0.12)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
save_fig(fig, "30_sigmoid_colormap_topdown_2d.png")


## Scene 9 - Run all exports in order

Execute all cells top-to-bottom once. The `renders/` folder will contain:

Intro (00-14): empty axes; `01_dataset_build.gif` holds no points briefly, then adds one point at a time in order `(3,2,1)` then `(2,3,0)` then shuffled rest, with many more frames on the first two points; fail point crosses unseen boundary; threshold shading/labels progression.

1. `00_axes_only.png`
2. `01_dataset_build.gif`
3. `02_fail_point_crosses_threshold.gif`
4. `03_threshold_unlabeled.png`
5. `04_threshold_green_right.png`
6. `05_threshold_red_left.png`
7. `06_threshold_micro_drift.gif`
8. `07_dataset_extra_pass_shifted_threshold.png`
9. `08_dataset_two_extra_no_threshold.png`
10. `09_threshold_labeled_st_eq_el.png`
11. `10_threshold_st_eq_el_dot_33.png`
12. `11_threshold_green_labeled_st_gt_el.png`
13. `12_threshold_red_labeled_st_lt_el.png`
14. `13_threshold_label_st_minus_el_eq_el_minus_el.png`
15. `14_threshold_label_st_minus_el_eq_0.png`
16. `15_dataset_overview.png`
17. `16_dataset_overview_labeled.png`
18. `17_parallel_lines_step_01.png` … `17_parallel_lines_step_05.png`
19. `18_parallel_lines_additive.gif`
20. `19_not_realistic_transition.gif`
21. `20_proportions_60_80_100.gif`
22. `21_between_60_and_80.png`
23. `22_exponential_mapping.gif`
24. `23_exponential_global.png`
25. `24_sigmoid_and_mirror.gif` … `30_sigmoid_colormap_topdown_2d.png`
